# دفتر التجربة: النموذج الأولي «جواب»

مقرر LDS-ADV-201 · المختبران 2.2 و2.4 والمشروع الختامي.

يسترجع «جواب» أقرب مقاطع قاعدة المعرفة لكل سؤال من فهرس متجهات في الذاكرة، ثم يصوغ مسودة إجابة، ثم يصحّحها مقابل الوسم، ويكتب سجل تشغيل في `runs/`. كل رقم في جدول نتائجك يعود إلى سجل تشغيل.

بلا مفتاح يصوغ المسودة **بديل استخراجي**: ينسخ أقرب جملة في المقطع الأول ويستشهد به. لا يترجم ولا يعيد الصياغة، فيُظهر أثر الاسترجاع والإعدادات وحدها. قل في جدولك أيهما أنتج الأرقام.

In [ ]:
# الإعداد: يعمل في Colab وفي JupyterLab وعلى حاسوب محمول دون تثبيت أي مكتبة
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("lds-adv-201").exists():
        os.system("git clone -q https://github.com/SDAIAAcademy/lds-adv-201")
    os.chdir("lds-adv-201/labs/spike")
else:
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "labs" / "spike" / "jawab.py").exists():
            os.chdir(p / "labs" / "spike")
            break
        if (p / "jawab.py").exists():
            os.chdir(p)
            break
sys.path.insert(0, str(Path.cwd()))
import jawab as J
print("المجلد:", Path.cwd().name, "· النموذج:", os.environ.get("LLM_MODEL") or "البديل الاستخراجي (لا نموذج لغوي)")

## 1 · النموذج

اختياري. الأرقام في الشرائح وفي مفتاح المدرّب من البديل الاستخراجي. النماذج المجانية لها حدود للطلبات في الدقيقة واليوم، والدفتر يحفظ كل رد في `cache/` فلا تتكرر الطلبات.

In [ ]:
# اختياري: نموذج لغوي عبر واجهة محادثة غير مرتبطة بمقدم خدمة بعينه
# في Colab أضف المفاتيح في لوحة Secrets، ثم أزل علامة التعليق من الأسطر الأربعة التالية:
# from google.colab import userdata
# os.environ["LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
# os.environ["LLM_MODEL"] = "google/gemma-4-31b-it:free"
# os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
print("نموذج لغوي" if J.using_model() else "البديل الاستخراجي: لا مفتاح، فكل رقم هنا رقم الاسترجاع والإعدادات وحدها")

## 2 · التشغيل المبدئي (المختبر 2.2)

الإعدادات كما سُلّمت: لا استبعاد، ولا احترام للصلاحيات، ولا امتناع.

In [ ]:
CASE = "enquiries"     # المشروع الختامي: "objections"
ROLE = "agent"         # المشروع الختامي: "officer"

starter = dict(J.STARTER, case=CASE, role=ROLE, label=f"{CASE}_starter")
rows_starter, fig_starter = J.run(starter)
J.show(fig_starter)
print("سجل التشغيل:", f"runs/{starter['label']}.csv", "و", f"runs/{starter['label']}.json")

## 3 · راجع بيدك

اقرأ الإجابات الخمس، واحكم عليها بمعيار الإجابة الصحيحة في ميثاقك قبل أن تنظر إلى حكم التصحيح الآلي.

In [ ]:
# خمس إجابات تراجعها بيدك قبل أن تثق بالتصحيح الآلي
import csv
labels = {q["qid"]: q for q in J.read_csv(J.CASES[CASE] / "questions.csv")}
for r in rows_starter[:5]:
    q = labels[r["qid"]]
    print(r["qid"], "|", r["question"])
    print("   الإجابة:", r["answer"])
    print("   المتوقع:", q["gold_fact"] or "امتناع", "من", q["gold_articles"] or "لا مصدر", "· التصحيح الآلي:", r["outcome"])

## 4 · التكييف وإعادة القياس (المختبر 2.4)

غيّر ثلاثة إعدادات على الأقل، وشغّل على الأسئلة الثلاثين نفسها.

In [ ]:
# التكييف (المختبر 2.4): الإعدادات المرجعية
adapted = dict(starter, label=f"{CASE}_adapted")
adapted["exclude_articles"] = ["A01", "A09", "A14", "A18", "A26", "A11"] if CASE == "enquiries" else ["B03", "B19"]
adapted["respect_permissions"] = True
adapted["chunking"] = "paragraph"
adapted["min_score"] = 0.25

rows_adapted, fig_adapted = J.run(adapted)
J.show(fig_adapted)
if CASE == "enquiries":
    assert fig_adapted["correct"] == 23, fig_adapted["correct"]

In [ ]:
# مقارنة مزدوجة: السؤال نفسه في التشغيلين
flips = J.compare(rows_starter, rows_adapted)
print("تحسّن:", flips["fixed"])
print("ساء:", flips["broken"])
print("صحيح في الاثنين:", flips["both_right"], "· خطأ في الاثنين:", flips["both_wrong"])

In [ ]:
# القرار مقابل العتبة التي كتبتها قبل التجربة (المختبر 1.3). غيّر الأرقام إلى عتبتك.
THRESHOLD = {"accuracy": 0.80, "lower_bound": 0.65}
f = fig_adapted
meets = (f["accuracy"] >= THRESHOLD["accuracy"] and f["wilson_low"] >= THRESHOLD["lower_bound"]
         and f["outcomes"].get("permission_leak", 0) == 0)
print(f"الدقة {f['accuracy']:.1%} · الحد الأدنى {f['wilson_low']:.1%} · تسريب الصلاحيات {f['outcomes'].get('permission_leak', 0)}")
print("يستوفي العتبة" if meets else "لا يستوفي العتبة: إعادة عمل أو توقف، بحسب أسباب الإخفاق")

## 5 · ما تحمله معك

ملفا التشغيلين في `runs/`: انسخ أرقامهما إلى جدول النتائج، وأرفق الملفين بدراسة الجدوى.